In [1]:
from stac2cube import get_stac_layers, missions
from stac2cube import interactive_time_view, generate_animation
import xarray as xr
import numpy as np
import dask
from dask.diagnostics import ProgressBar
import rioxarray
import matplotlib.pyplot as plt
import pandas as pd

## Lazy

In [2]:
mission = "s2"                              # both name and allias work. Can not be None!
polygon = "./polygons/test.gpkg"            # if not available, try optional cell below
resolution = 10                             # if None -> default_resolution
daterange = ["2025-06-01", "2025-06-10"]    # if None -> every available date
bands = ["blue", "green", "red", "nir"]     # if None -> all bands of the relevant mission, but also SCL
max_cc = 100                                # 0-100, if None -> 100
clip_raster = False                         # if None -> False
cloud_masking = False                       # if None -> False
indices = ["ndvi", "ndwi"]                  # if None -> no indices
output = None                               # Will return lazy! To export, replace with path to netcdf.
stats = None                                # if None -> no stats
aggregator = None                           # if None -> no aggregation

In [3]:
stac = get_stac_layers(mission=mission, polygon=polygon, resolution=resolution, 
                            daterange=daterange, bands=bands, max_cc=max_cc, 
                            clip_raster=clip_raster, cloud_masking=cloud_masking, indices=indices, 
                            output=output, aggregator=aggregator, stats=stats
                            )

<xarray.DataArray 'Spectral_Temporal_Stack' (time: 5, band: 6, y: 247, x: 511)> Size: 30MB
dask.array<transpose, shape=(5, 6, 247, 511), dtype=float64, chunksize=(1, 1, 247, 511), chunktype=numpy.ndarray>
Coordinates:
  * time         (time) datetime64[ns] 40B 2025-06-02 2025-06-05 ... 2025-06-10
  * band         (band) <U5 120B 'blue' 'green' 'red' 'nir' 'ndvi' 'ndwi'
  * y            (y) float64 2kB 4.547e+06 4.547e+06 ... 4.544e+06 4.544e+06
  * x            (x) float64 4kB 6.67e+05 6.67e+05 ... 6.721e+05 6.721e+05
    spatial_ref  int32 4B 0
Attributes:
    nodata:          0
    indices:         ['ndvi', 'ndwi']
    spectral_bands:  ['blue', 'green', 'red', 'nir']
    mission:         sentinel_2_l2a
    tile_id:         ['35TPF']
    bbox:            [28.98752355404842, 41.03150270425637, 29.04759458831022...
    crs:             WGS 84 / UTM zone 35N
    transform:       | 10.00, 0.00, 667020.00|\n| 0.00,-10.00, 4546620.00|\n|...


## Computed

In [6]:
img = ("./results/coregistered_naryn.nc")

dataset = xr.open_dataset(img)
stac = dataset.Spectral_Temporal_Stack

## Interactive tools

In [10]:
interactive_time_view(stac=stac, display_mode='rgb', widget_type='slider')

IntSlider(value=0, description='Time', layout=Layout(width='800px'), max=39)

Output()

interactive(children=(IntSlider(value=0, description='Time', layout=Layout(width='800px'), max=39), Output()),…

In [7]:
interactive_time_view(stac=stac, display_mode='rgb', widget_type='dropdown')

Dropdown(description='Date:', layout=Layout(width='300px'), options=(('25-01-2024', 0), ('24-02-2024', 1), ('2…

Output()

interactive(children=(Dropdown(description='Date:', layout=Layout(width='300px'), options=(('25-01-2024', 0), …

In [8]:
generate_animation(
    stac,
    output_path="./animations/test.gif",
    display_mode="rgb",
    frame_interval_ms=400,
)

GIF saved: ./animations/test.gif  (frames=40, colors=128, max_size=900)


In [2]:
img1 = ("./results/animation_nonregistered.nc")
img2 = ("./results/animation_coregistered_sr.nc")
dataset = xr.open_dataset(img1)
stac = dataset.Spectral_Temporal_Stack
dataset2 = xr.open_dataset(img2)
stac2 = dataset2.Spectral_Temporal_Stack



generate_animation_mp4_max(stac2, "./animations/sr.mp4", "rgb", frame_interval_ms=400)
generate_animation_mp4_max(stac, "./animations/normal.mp4", "rgb", frame_interval_ms=400)


OSError: [Errno 22] Invalid argument

FFMPEG COMMAND:
c:\Users\baa26kk\micromamba\envs\stac2cubetester\Library\bin\ffmpeg.exe -y -f rawvideo -vcodec rawvideo -s 898x338 -pix_fmt rgb24 -r 2.00 -i - -an -vcodec libx264 -pix_fmt yuv444p -v warning -crf 0 -preset veryslow -x264-params deblock=0:0 c:\Users\baa26kk\Desktop\terrabyte_globus\stac2cube\interactive\animations\sr.mp4

FFMPEG STDERR OUTPUT:


In [3]:
from stac2cube import clip_stac
import xarray as xr

In [4]:
img = ("./results/non_coregistered_naryn.nc")

dataset = xr.open_dataset(img)
stac = dataset.Spectral_Temporal_Stack

stac_clipped = clip_stac(stac, polygon="./polygons/animation_clip.gpkg", crs = "EPSG:32643")

In [15]:
from stac2cube import cloud_filter
stac_final = cloud_filter(stac_clipped, 0)

In [5]:
stac_clipped

<xarray.DataArray 'Spectral_Temporal_Stack' (time: 40, band: 6, y: 151, x: 277)> Size: 80MB
array([[[[ 7.92600000e-01,  7.83800000e-01,  7.63100000e-01, ...,
           7.75400000e-01,  7.81700000e-01,  7.85400000e-01],
         [ 8.03700000e-01,  8.10200000e-01,  7.98300000e-01, ...,
           7.72700000e-01,  7.91700000e-01,  7.87500000e-01],
         [ 6.05600000e-01,  6.28200000e-01,  7.15400000e-01, ...,
           8.07300000e-01,  7.84900000e-01,  7.69300000e-01],
         ...,
         [ 8.07400000e-01,  8.39100000e-01,  8.22500000e-01, ...,
           8.18100000e-01,  8.24100000e-01,  8.18400000e-01],
         [ 7.48100000e-01,  8.35400000e-01,  7.96300000e-01, ...,
           8.32700000e-01,  8.29600000e-01,  8.19300000e-01],
         [ 7.69700000e-01,  8.13800000e-01,  6.83400000e-01, ...,
           8.31200000e-01,  8.31900000e-01,  8.16600000e-01]],

        [[ 7.94300000e-01,  7.90100000e-01,  7.87000000e-01, ...,
           8.00700000e-01,  8.15700000e-01,  8.03800000e-01],
         [ 8.26200000e-01,  8.11100000e-01,  8.05700000e-01, ...,
           8.15900000e-01,  8.14300000e-01,  8.17300000e-01],
         [ 6.08200000e-01,  6.45900000e-01,  7.43500000e-01, ...,
           8.50300000e-01,  8.05600000e-01,  7.98100000e-01],
...
         [ 1.37869972e-01,  1.26101183e-01,  1.17155939e-01, ...,
           7.60025873e-02,  7.48528175e-02,  1.32702397e-02],
         [ 2.19076006e-01,  1.60867373e-01,  1.28159906e-01, ...,
           7.19039117e-02,  7.33343497e-02,  3.45840868e-02],
         [ 1.93967759e-01,  1.55249934e-01,  1.22349103e-01, ...,
           6.32708634e-02,  9.70078740e-02,  4.79628675e-02]],

        [[-2.17833285e-02, -3.07692308e-03, -3.58142962e-02, ...,
          -1.90348525e-01, -1.70710383e-01, -1.71784602e-01],
         [ 2.80808963e-02,  2.48202902e-02,  1.28555799e-02, ...,
          -1.77759056e-01, -1.71044202e-01, -1.95045045e-01],
         [ 7.01058201e-02,  6.70603121e-02,  8.36447767e-03, ...,
          -2.20476190e-01, -2.23873442e-01, -2.48663782e-01],
         ...,
         [-2.65260197e-01, -2.65987550e-01, -2.58495146e-01, ...,
          -1.69009136e-01, -1.39375929e-01, -4.83184334e-02],
         [-3.38058888e-01, -2.97998308e-01, -2.51385719e-01, ...,
          -1.63495419e-01, -1.41559916e-01, -8.19052121e-02],
         [-3.38775510e-01, -2.76446523e-01, -2.07017544e-01, ...,
          -1.46657454e-01, -1.89955586e-01, -8.64857639e-02]]]])
Coordinates:
  * time              (time) datetime64[ns] 320B 2024-01-25 ... 2024-12-25
  * band              (band) <U5 120B 'blue' 'green' 'red' 'nir' 'ndvi' 'ndwi'
  * y                 (y) float64 1kB 4.581e+06 4.581e+06 ... 4.579e+06
  * x                 (x) float64 2kB 5.436e+05 5.436e+05 ... 5.464e+05
    cloud_percentage  (time) int64 320B ...
    spatial_ref       int32 4B 0

In [6]:
stac_clipped.to_netcdf("./results/clipped_nonregistered.nc")

In [ ]:
from pathlib import Path
import numpy as np
from PIL import Image
import imageio.v3 as iio

# ----------------------------
# USER SETTINGS
# ----------------------------
INPUT_DIR = Path("C:\\Users\\baa26kk\\Desktop\\terrabyte_globus\\stac2cube\\assets")          # folder containing a1.png ... b3.png
OUT_DIR = Path("C:\\Users\\baa26kk\\Desktop\\terrabyte_globus\\stac2cube\\assets")            # output folder
OUT_DIR.mkdir(exist_ok=True)

# Animation feel (tweak these)
FPS = 12                 # 10–15 is usually nice for "slow but not too slow"
HOLD_SECONDS = 0.8       # how long each image stays fully visible
FADE_SECONDS = 0.6       # crossfade time between images

# Output formats
MAKE_GIF = True
MAKE_MP4 = True

# Filenames (your sequences)
SEQ_A = ["a1.png", "a2.png", "a3.png"]
SEQ_B = ["b1.png", "b2.png", "b3.png"]

# ----------------------------
# FUNCTIONS
# ----------------------------
def load_rgb(path: Path) -> np.ndarray:
    """Load image as RGB numpy array."""
    img = Image.open(path).convert("RGB")
    return np.array(img)

def ensure_same_shape(frames: list[np.ndarray]) -> list[np.ndarray]:
    """Ensure all frames are the same size (resize to first frame if needed)."""
    h0, w0 = frames[0].shape[:2]
    out = [frames[0]]
    for f in frames[1:]:
        if f.shape[:2] != (h0, w0):
            f = np.array(Image.fromarray(f).resize((w0, h0), Image.LANCZOS))
        out.append(f)
    return out

def make_crossfade_frames(images: list[np.ndarray], fps: int, hold_s: float, fade_s: float) -> list[np.ndarray]:
    """Create frames with hold + crossfade between consecutive images."""
    hold_frames = max(1, int(round(hold_s * fps)))
    fade_frames = max(1, int(round(fade_s * fps)))

    frames = []
    for i in range(len(images)):
        # hold current image
        for _ in range(hold_frames):
            frames.append(images[i])

        # fade to next image
        if i < len(images) - 1:
            a = images[i].astype(np.float32)
            b = images[i + 1].astype(np.float32)

            for t in range(1, fade_frames + 1):
                alpha = t / fade_frames  # 0->1
                blended = (1 - alpha) * a + alpha * b
                frames.append(blended.astype(np.uint8))

    return frames

def save_gif(frames: list[np.ndarray], out_path: Path, fps: int):
    # duration per frame in ms for GIF
    duration_ms = int(round(1000 / fps))
    iio.imwrite(out_path, frames, duration=duration_ms, loop=0)

def save_mp4(frames: list[np.ndarray], out_path: Path, fps: int):
    # H.264 MP4 (works nicely for GitHub attachments)
    iio.imwrite(out_path, frames, fps=fps, codec="libx264", quality=8, pixelformat="yuv420p")

def build_animation(seq: list[str], name: str):
    imgs = [load_rgb(INPUT_DIR / s) for s in seq]
    imgs = ensure_same_shape(imgs)
    frames = make_crossfade_frames(imgs, FPS, HOLD_SECONDS, FADE_SECONDS)

    if MAKE_GIF:
        save_gif(frames, OUT_DIR / f"{name}.gif", FPS)
        print(f"✅ Saved: {OUT_DIR / f'{name}.gif'}")

    if MAKE_MP4:
        save_mp4(frames, OUT_DIR / f"{name}.mp4", FPS)
        print(f"✅ Saved: {OUT_DIR / f'{name}.mp4'}")


# ----------------------------
# RUN
# ----------------------------
if __name__ == "__main__":
    build_animation(SEQ_A, "cube_before")  # a1→a2→a3
    build_animation(SEQ_B, "cube_after")   # b1→b2→b3


✅ Saved: C:\Users\baa26kk\Desktop\terrabyte_globus\stac2cube\assets\cube_before.gif


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3507, 2480) to (3520, 2480) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


✅ Saved: C:\Users\baa26kk\Desktop\terrabyte_globus\stac2cube\assets\cube_before.mp4


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3507, 2480) to (3520, 2480) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


✅ Saved: C:\Users\baa26kk\Desktop\terrabyte_globus\stac2cube\assets\cube_after.gif
✅ Saved: C:\Users\baa26kk\Desktop\terrabyte_globus\stac2cube\assets\cube_after.mp4


In [7]:
from pathlib import Path
import numpy as np
from PIL import Image
import imageio.v3 as iio

INPUT_DIR = Path("C:\\Users\\baa26kk\\Desktop\\terrabyte_globus\\stac2cube\\assets")          # folder containing a1.png ... b3.png
OUT_DIR = Path("C:\\Users\\baa26kk\\Desktop\\terrabyte_globus\\stac2cube\\assets")            # output folder
#OUT_DIR.mkdir(exist_ok=True)
# ✅ better GitHub-friendly settings
FPS = 10
HOLD_SECONDS = 0.5
FADE_SECONDS = 0.4

# ✅ resize (HUGE size reduction)
RESIZE_WIDTH = 1400   # try 1200 / 1400 / 1600

SEQ_A = ["a1.png", "a2.png", "a3.png"]
SEQ_B = ["b1.png", "b2.png", "b3.png"]


def load_rgb_resized(path: Path) -> np.ndarray:
    img = Image.open(path).convert("RGB")
    if RESIZE_WIDTH is not None:
        w, h = img.size
        new_w = RESIZE_WIDTH
        new_h = int(h * (new_w / w))
        img = img.resize((new_w, new_h), Image.LANCZOS)
    return np.array(img)


def make_crossfade_frames(images, fps, hold_s, fade_s):
    hold_frames = max(1, int(round(hold_s * fps)))
    fade_frames = max(1, int(round(fade_s * fps)))

    frames = []
    for i in range(len(images)):
        for _ in range(hold_frames):
            frames.append(images[i])

        if i < len(images) - 1:
            a = images[i].astype(np.float32)
            b = images[i + 1].astype(np.float32)

            for t in range(1, fade_frames + 1):
                alpha = t / fade_frames
                blended = (1 - alpha) * a + alpha * b
                frames.append(blended.astype(np.uint8))

    return frames


def save_mp4(frames, out_path: Path, fps: int):
    # ✅ CRF-style compression (size-controlled)
    iio.imwrite(
        out_path,
        frames,
        fps=fps,
        codec="libx264",
        pixelformat="yuv420p",
        ffmpeg_params=["-crf", "26", "-preset", "slow", "-movflags", "+faststart"],
    )


def build_animation(seq, name):
    imgs = [load_rgb_resized(INPUT_DIR / s) for s in seq]
    frames = make_crossfade_frames(imgs, FPS, HOLD_SECONDS, FADE_SECONDS)
    save_mp4(frames, OUT_DIR / f"{name}.mp4", FPS)
    print(f"✅ Saved: {OUT_DIR / f'{name}.mp4'}")


if __name__ == "__main__":
    build_animation(SEQ_A, "cube_before")
    build_animation(SEQ_B, "cube_after")


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (1400, 990) to (1408, 992) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


✅ Saved: C:\Users\baa26kk\Desktop\terrabyte_globus\stac2cube\assets\cube_before.mp4


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (1400, 990) to (1408, 992) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


✅ Saved: C:\Users\baa26kk\Desktop\terrabyte_globus\stac2cube\assets\cube_after.mp4


In [8]:
from pathlib import Path
import numpy as np
from PIL import Image
import imageio.v3 as iio

# ----------------------------
# USER SETTINGS
# ----------------------------
INPUT_DIR = Path("C:\\Users\\baa26kk\\Desktop\\terrabyte_globus\\stac2cube\\assets")          # folder containing a1.png ... b3.png
OUT_DIR = Path("C:\\Users\\baa26kk\\Desktop\\terrabyte_globus\\stac2cube\\assets")  

SEQ_A = ["a1.png", "a2.png", "a3.png"]
SEQ_B = ["b1.png", "b2.png", "b3.png"]

# Feel (tune these)
FPS = 10
HOLD_SECONDS = 0.6
FADE_SECONDS = 0.4

# Size control (most important!)
RESIZE_WIDTH = 1200   # try 900 / 1200 / 1400 depending on how big you want in README

# Optional: reduce colors to shrink GIF a lot (good for satellite images)
COLOR_LIMIT = 256     # 256 max (classic GIF). Try 128 if you want smaller.

# ----------------------------
# HELPERS
# ----------------------------
def load_rgb_resized(path: Path) -> Image.Image:
    img = Image.open(path).convert("RGB")

    if RESIZE_WIDTH is not None:
        w, h = img.size
        new_w = RESIZE_WIDTH
        new_h = int(h * (new_w / w))
        img = img.resize((new_w, new_h), Image.LANCZOS)

    return img

def crossfade_frames(img_a: Image.Image, img_b: Image.Image, n: int):
    frames = []
    for i in range(1, n + 1):
        alpha = i / n
        frames.append(Image.blend(img_a, img_b, alpha))
    return frames

def build_gif_sequence(files, fps, hold_s, fade_s):
    hold_frames = max(1, int(round(hold_s * fps)))
    fade_frames = max(1, int(round(fade_s * fps)))

    imgs = [load_rgb_resized(INPUT_DIR / f) for f in files]

    frames = []
    for i in range(len(imgs)):
        # Hold
        frames.extend([imgs[i]] * hold_frames)

        # Fade to next
        if i < len(imgs) - 1:
            frames.extend(crossfade_frames(imgs[i], imgs[i + 1], fade_frames))

    return frames

def save_gif(frames, out_path: Path, fps: int, color_limit: int):
    duration_ms = int(round(1000 / fps))

    # Convert to palette mode to shrink file size
    # (quantization happens here; it’s the main GIF compression step)
    pal_frames = []
    for fr in frames:
        pal_frames.append(fr.convert("P", palette=Image.Palette.ADAPTIVE, colors=color_limit))

    pal_frames[0].save(
        out_path,
        save_all=True,
        append_images=pal_frames[1:],
        duration=duration_ms,
        loop=0,           # ✅ infinite loop
        optimize=True,
        disposal=2
    )
    print(f"✅ Saved GIF: {out_path}")

# ----------------------------
# RUN
# ----------------------------
if __name__ == "__main__":
    frames_before = build_gif_sequence(SEQ_A, FPS, HOLD_SECONDS, FADE_SECONDS)
    save_gif(frames_before, OUT_DIR / "cube_before.gif", FPS, COLOR_LIMIT)

    frames_after = build_gif_sequence(SEQ_B, FPS, HOLD_SECONDS, FADE_SECONDS)
    save_gif(frames_after, OUT_DIR / "cube_after.gif", FPS, COLOR_LIMIT)


✅ Saved GIF: C:\Users\baa26kk\Desktop\terrabyte_globus\stac2cube\assets\cube_before.gif
✅ Saved GIF: C:\Users\baa26kk\Desktop\terrabyte_globus\stac2cube\assets\cube_after.gif


In [ ]:
from pathlib import Path
from PIL import Image

# ----------------------------
# SETTINGS
# ----------------------------
INPUT_DIR = Path("C:\\Users\\baa26kk\\Desktop\\terrabyte_globus\\stac2cube\\assets")          # folder containing a1.png ... b3.png
OUT_DIR = Path("C:\\Users\\baa26kk\\Desktop\\terrabyte_globus\\stac2cube\\assets")  

SEQ_A = ["a1.png", "a2.png", "a3.png"]
SEQ_B = ["b1.png", "b2.png", "b3.png"]

RESIZE_WIDTH = 700          # 700–1200 is realistic for GitHub GIFs
HOLD_MS = 700               # how long each scene stays
FADE_FRAMES = 3             # keep small! (2–4)
FADE_MS_TOTAL = 300         # total fade time

COLOR_LIMIT = 64            # 256=best quality, 128 good, 64 small, 32 very small


def load_resized(path: Path) -> Image.Image:
    img = Image.open(path).convert("RGB")
    w, h = img.size
    new_w = RESIZE_WIDTH
    new_h = int(h * (new_w / w))
    return img.resize((new_w, new_h), Image.LANCZOS)


def crossfade(a: Image.Image, b: Image.Image, n: int):
    return [Image.blend(a, b, (i + 1) / n) for i in range(n)]


def make_frames_and_durations(files):
    imgs = [load_resized(INPUT_DIR / f) for f in files]

    frames = []
    durations = []

    # Main frames + fades
    for i in range(len(imgs)):
        frames.append(imgs[i])
        durations.append(HOLD_MS)

        if i < len(imgs) - 1:
            fades = crossfade(imgs[i], imgs[i + 1], FADE_FRAMES)
            fade_ms = max(1, FADE_MS_TOTAL // FADE_FRAMES)
            frames.extend(fades)
            durations.extend([fade_ms] * len(fades))

    # Quantize frames (this is where GIF size really shrinks)
    frames_q = [
        fr.convert("P", palette=Image.Palette.ADAPTIVE, colors=COLOR_LIMIT)
        for fr in frames
    ]

    return frames_q, durations


def save_gif(files, out_name):
    frames, durations = make_frames_and_durations(files)
    out_path = OUT_DIR / out_name

    frames[0].save(
        out_path,
        save_all=True,
        append_images=frames[1:],
        duration=durations,  # ✅ list of durations per frame (no duplicated hold frames)
        loop=0,              # ✅ infinite loop
        optimize=True,
        disposal=2,
    )

    print(f"✅ Saved: {out_path}")


if __name__ == "__main__":
    save_gif(SEQ_A, "cube_before2.gif")
    save_gif(SEQ_B, "cube_after2.gif")


✅ Saved: C:\Users\baa26kk\Desktop\terrabyte_globus\stac2cube\assets\cube_before2.gif
✅ Saved: C:\Users\baa26kk\Desktop\terrabyte_globus\stac2cube\assets\cube_after2.gif


In [15]:
from pathlib import Path
from PIL import Image

# ----------------------------
# SETTINGS
# ----------------------------
INPUT_DIR = Path(r"C:\Users\baa26kk\Desktop\terrabyte_globus\stac2cube\interactive\results\cogs\non_registered\maps")
OUT_DIR   = Path(r"C:\Users\baa26kk\Desktop\terrabyte_globus\stac2cube\assets")

OUT_NAME = "cube_before.gif"

RESIZE_WIDTH = 700
HOLD_MS = 80          # ~10–12 seconds total for 34 frames (fast flow)
FADE_FRAMES = 2
FADE_MS_TOTAL = 120
COLOR_LIMIT = 64


def load_resized(path: Path) -> Image.Image:
    img = Image.open(path).convert("RGB")
    w, h = img.size
    new_w = RESIZE_WIDTH
    new_h = int(h * (new_w / w))
    return img.resize((new_w, new_h), Image.LANCZOS)


def crossfade(a: Image.Image, b: Image.Image, n: int):
    return [Image.blend(a, b, (i + 1) / n) for i in range(n)]


def numeric_key(p: Path) -> int:
    # expects filenames like x1.png, x34.png
    stem = p.stem  # "x34"
    return int(stem[1:])  # 34


def make_frames_and_durations(files):
    imgs = [load_resized(f) for f in files]

    frames = []
    durations = []

    for i in range(len(imgs)):
        frames.append(imgs[i])
        durations.append(HOLD_MS)

        if i < len(imgs) - 1:
            fades = crossfade(imgs[i], imgs[i + 1], FADE_FRAMES)
            fade_ms = max(1, FADE_MS_TOTAL // FADE_FRAMES)
            frames.extend(fades)
            durations.extend([fade_ms] * len(fades))

    frames_q = [
        fr.convert("P", palette=Image.Palette.ADAPTIVE, colors=COLOR_LIMIT)
        for fr in frames
    ]

    return frames_q, durations


def save_gif(frames, durations, out_path: Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    frames[0].save(
        out_path,
        save_all=True,
        append_images=frames[1:],
        duration=durations,
        loop=0,
        optimize=True,
        disposal=2,
    )
    print(f"✅ Saved: {out_path}")


if __name__ == "__main__":
    # Collect x1.png ... x34.png
    files = sorted(INPUT_DIR.glob("y*.png"), key=numeric_key)

    if not files:
        raise FileNotFoundError(f"No files matching x*.png found in: {INPUT_DIR}")

    # (Optional) sanity check expected count
    print(f"Found {len(files)} PNGs. First: {files[0].name}, Last: {files[-1].name}")

    frames, durations = make_frames_and_durations(files)
    save_gif(frames, durations, OUT_DIR / OUT_NAME)


Found 35 PNGs. First: y1.png, Last: y35.png
✅ Saved: C:\Users\baa26kk\Desktop\terrabyte_globus\stac2cube\assets\cube_before.gif


In [16]:
from pathlib import Path
from PIL import Image

# ----------------------------
# SETTINGS
# ----------------------------
INPUT_DIR = Path(r"C:\Users\baa26kk\Desktop\terrabyte_globus\stac2cube\interactive\results\cogs\coregistered_sr\maps")
OUT_DIR   = Path(r"C:\Users\baa26kk\Desktop\terrabyte_globus\stac2cube\assets")

OUT_NAME = "cube_after.gif"

RESIZE_WIDTH = 700
HOLD_MS = 80          # ~10–12 seconds total for 34 frames (fast flow)
FADE_FRAMES = 2
FADE_MS_TOTAL = 120
COLOR_LIMIT = 64


def load_resized(path: Path) -> Image.Image:
    img = Image.open(path).convert("RGB")
    w, h = img.size
    new_w = RESIZE_WIDTH
    new_h = int(h * (new_w / w))
    return img.resize((new_w, new_h), Image.LANCZOS)


def crossfade(a: Image.Image, b: Image.Image, n: int):
    return [Image.blend(a, b, (i + 1) / n) for i in range(n)]


def numeric_key(p: Path) -> int:
    # expects filenames like x1.png, x34.png
    stem = p.stem  # "x34"
    return int(stem[1:])  # 34


def make_frames_and_durations(files):
    imgs = [load_resized(f) for f in files]

    frames = []
    durations = []

    for i in range(len(imgs)):
        frames.append(imgs[i])
        durations.append(HOLD_MS)

        if i < len(imgs) - 1:
            fades = crossfade(imgs[i], imgs[i + 1], FADE_FRAMES)
            fade_ms = max(1, FADE_MS_TOTAL // FADE_FRAMES)
            frames.extend(fades)
            durations.extend([fade_ms] * len(fades))

    frames_q = [
        fr.convert("P", palette=Image.Palette.ADAPTIVE, colors=COLOR_LIMIT)
        for fr in frames
    ]

    return frames_q, durations


def save_gif(frames, durations, out_path: Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    frames[0].save(
        out_path,
        save_all=True,
        append_images=frames[1:],
        duration=durations,
        loop=0,
        optimize=True,
        disposal=2,
    )
    print(f"✅ Saved: {out_path}")


if __name__ == "__main__":
    # Collect x1.png ... x34.png
    files = sorted(INPUT_DIR.glob("x*.png"), key=numeric_key)

    if not files:
        raise FileNotFoundError(f"No files matching x*.png found in: {INPUT_DIR}")

    # (Optional) sanity check expected count
    print(f"Found {len(files)} PNGs. First: {files[0].name}, Last: {files[-1].name}")

    frames, durations = make_frames_and_durations(files)
    save_gif(frames, durations, OUT_DIR / OUT_NAME)


Found 35 PNGs. First: x1.png, Last: x35.png
✅ Saved: C:\Users\baa26kk\Desktop\terrabyte_globus\stac2cube\assets\cube_after.gif
